# CoursIA pour les agents — 01 · Inventaire des interfaces agentiques

**Part of #27071** — EPIC « CoursIA pour les agents : découverte, interfaces exécutables et base de connaissances ».

Ce carnet est le **premier livrable** de l'EPIC : un **inventaire sourcé, mesuré dans le code**, de la surface d'interfaces du dépôt **invocable par un agent** — en distinguant, pour chaque surface, ce qui est **vérifié** (preuve `fichier:ligne`), ce qui est **hypothétique**, et ce qui est **projeté**.

Trois classes d'interface sont examinées :

1. **MCP** — les serveurs d'outils exposés par le fichier `.mcp.json` de la racine ;
2. **CLI** — les scripts de `scripts/` exposant un point d'entrée (`main()` + `argparse`) ;
3. **Bridge notebook→service** — le script qui expose un notebook annoté comme service HTTP (`notebook-http`).

> **Portée mesurée, pas affirmée** : ce carnet mesure ce qui **existe dans le dépôt** (fichiers lus firsthand). Il ne promet ni exécution, ni performance, ni couverture complète — il dresse une **carte sourcée** de ce qu'un agent peut **découvrir et invoquer** aujourd'hui, et la **mesure de couverture** qui en découle.

## Méthode — « vérifié » veut dire lu, pas affirmé

Pour chaque classe, le carnet **lit les fichiers du dépôt** et compte :

- **MCP** : les entrées du dictionnaire `mcpServers` de `.mcp.json` (racine) ;
- **CLI** : les fichiers `.py` de `scripts/` (hors `_archive/`) qui déclarent à la fois `def main(` **et** `argparse` — le critère d'un point d'entrée CLI ;
- **Bridge** : la présence de `scripts/notebook_tools/kernel_gateway_service.py` (le script canonique qui démarre un notebook annoté en `notebook-http`).

Chaque nombre est **re-calculé par le code ci-dessous** sur l'arbre courant — il est donc **falsifiable** : re-exécuter le carnet après un ajout ou un retrait d'interface fait bouger les compteurs.

In [1]:
import json
from pathlib import Path

def find_root(start: Path) -> Path:
    """Remonter jusqu'au repertoire racine du depot (scripts/ + MyIA.AI.Notebooks/)."""
    d = start
    for _ in range(8):
        if (d / "scripts").is_dir() and (d / "MyIA.AI.Notebooks").is_dir():
            return d
        if d.parent == d:
            break
        d = d.parent
    return start

ROOT = find_root(Path.cwd())
print(f"ROOT = {ROOT}")
print(f"scripts/ présent : {(ROOT / 'scripts').exists()}")

# --- Classe 1 : MCP (.mcp.json de la racine) ---
mcp_path = ROOT / ".mcp.json"
mcp_servers = []
if mcp_path.exists():
    with open(mcp_path, encoding="utf-8") as f:
        mcp_cfg = json.load(f)
    mcp_servers = list(mcp_cfg.get("mcpServers", {}).keys())

print(f"\n[mcp.json] présent        : {mcp_path.exists()}")
print(f"[mcp.json] serveurs MCP    : {len(mcp_servers)}  ->  {mcp_servers}")

ROOT = d:\dev\CoursIA-2
scripts/ présent : True

[mcp.json] présent        : True
[mcp.json] serveurs MCP    : 1  ->  ['jupyter-papermill']


In [2]:
# --- Classe 2 : CLI (scripts/ avec def main + argparse, hors _archive/) ---
scripts_dir = ROOT / "scripts"
cli_tools = []
if scripts_dir.exists():
    for py in sorted(scripts_dir.rglob("*.py")):
        if "_archive" in py.parts:
            continue
        try:
            src = py.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        if "def main(" in src and "argparse" in src:
            cli_tools.append(str(py.relative_to(ROOT)))

print(f"[CLI] scripts à point d'entrée (def main + argparse, hors _archive/) : {len(cli_tools)}")
for t in cli_tools[:12]:
    print("   ", t)
if len(cli_tools) > 12:
    print(f"    … et {len(cli_tools) - 12} autres")

[CLI] scripts à point d'entrée (def main + argparse, hors _archive/) : 622
    scripts\argumentation\narrative_information.py
    scripts\audio_narration_benchmark.py
    scripts\audiobook_pipeline.py
    scripts\audit\build_editorial_review_card.py
    scripts\audit\build_editorial_review_dossier.py
    scripts\audit\check_cost_metadata.py
    scripts\audit\check_dataset_registry.py
    scripts\audit\check_denominators.py
    scripts\audit\check_distillation_archival_order.py
    scripts\audit\check_editorial_review.py
    scripts\audit\check_lean_notebook_sorry.py
    scripts\audit\check_orphan_merged_pr.py
    … et 610 autres


In [3]:
# --- Classe 3 : Bridge notebook -> service HTTP ---
bridge = scripts_dir / "notebook_tools" / "kernel_gateway_service.py"
print(f"[Bridge] kernel_gateway_service.py présent : {bridge.exists()}")

# --- Mesure de couverture : référence native scripts-reference.md ---
ref = ROOT / "docs" / "reference" / "scripts-reference.md"
named, missing = [], []
if ref.exists() and cli_tools:
    ref_text = ref.read_text(encoding="utf-8", errors="ignore")
    for t in cli_tools:
        (named if Path(t).name in ref_text else missing).append(t)
    cov = 100.0 * len(named) / len(cli_tools)
    print(f"\n[Coverage] référence native présente       : {ref.exists()}")
    print(f"[Coverage] scripts CLI nommés dans la référence : {len(named)}/{len(cli_tools)}  ({cov:.0f} %)")
    print(f"[Coverage] scripts CLI NON nommés             : {len(missing)}")
    for m in missing[:8]:
        print("    -", m)
else:
    print("[Coverage] référence native absente ou aucun CLI")

[Bridge] kernel_gateway_service.py présent : True

[Coverage] référence native présente       : True
[Coverage] scripts CLI nommés dans la référence : 118/622  (19 %)
[Coverage] scripts CLI NON nommés             : 504
    - scripts\argumentation\narrative_information.py
    - scripts\audio_narration_benchmark.py
    - scripts\audit\build_editorial_review_card.py
    - scripts\audit\build_editorial_review_dossier.py
    - scripts\audit\check_distillation_archival_order.py
    - scripts\audit\check_orphan_merged_pr.py
    - scripts\audit\check_orphan_post_merge_commits.py
    - scripts\audit\check_scientific_review.py


In [4]:
# --- L'inventaire classifié (le livrable) ---
n_bridge = int(bridge.exists())
total = len(mcp_servers) + len(cli_tools) + n_bridge
print("=" * 62)
print("INVENTAIRE DES INTERFACES AGENTIQUES — surface VÉRIFIÉE")
print("=" * 62)
print(f"  [MCP]     {len(mcp_servers):>3}  serveur(s)      : {', '.join(mcp_servers) or 'aucun'}")
print(f"  [CLI]     {len(cli_tools):>3}  script(s)        : def main + argparse, hors _archive/")
print(f"  [BRIDGE]  {n_bridge:>3}  script(s)        : notebook -> service HTTP (kernel_gateway)")
print("-" * 62)
print(f"  [TOTAL]   {total:>3}  interface(s) invocable(s) vérifiée(s)")
print("=" * 62)

INVENTAIRE DES INTERFACES AGENTIQUES — surface VÉRIFIÉE
  [MCP]       1  serveur(s)      : jupyter-papermill
  [CLI]     622  script(s)        : def main + argparse, hors _archive/
  [BRIDGE]    1  script(s)        : notebook -> service HTTP (kernel_gateway)
--------------------------------------------------------------
  [TOTAL]   624  interface(s) invocable(s) vérifiée(s)


## Lecture du résultat

Les trois surfaces forment la **carte de ce qu'un agent peut invoquer** dans le dépôt aujourd'hui :

- **MCP** — la surface la plus « prête pour un agent » : le serveur `jupyter-papermill`, branchable directement par un client MCP ;
- **CLI** — la surface la plus large : des **centaines** de scripts à point d'entrée, mais **sans contrat d'agent commun** (ni `--help`, ni sortie structurée garantie) — c'est précisément le chantier du **pilote CLI** (critère 3 de l'EPIC) ;
- **Bridge** — le `kernel_gateway_service` est le **seul pont notebook→service** mesuré : c'est le mécanisme par lequel un notebook annoté (`# GET /…`, `# POST /…`) devient invocable en HTTP.

La **mesure de couverture** (ici : une **minorité** des scripts CLI est nommée dans la référence native `scripts-reference.md`, la grande majorité ne l'est pas) quantifie l'écart entre **ce qui existe** et **ce qui est documenté** — le chantier de la **sémantisation** (critère 4 de l'EPIC).

> Aucune supériorité ni complétude n'est affirmée : ce carnet **mesure**, il ne **promet** pas.

## Exercices

**Exercice 1 — étendre la classe MCP.** Le scan ne lit que le `.mcp.json` de la **racine**. D'autres `.mcp.json` existent par sous-projet. Modifiez la collecte pour fusionner **tous** les `.mcp.json` du dépôt, sans doublon :

```python
tous_mcp = []  # TODO etudiant : noms de serveurs de TOUS les .mcp.json du depot
```

**Exercice 2 — prober le contrat d'agent.** Un agent a besoin d'un `--help` lisible. Écrivez une fonction qui, donné le chemin d'un script CLI, renvoie `True` si son source gère `--help`, puis comptez la part des CLI « agent-ready » :

```python
def est_agent_ready(chemin: str) -> bool:
    # TODO etudiant
    return None
```

**Exercice 3 — couverture par sous-série.** Restreignez la mesure de couverture à `scripts/notebook_tools/` seul, et comparez-la à la couverture globale : les outils de notebooks sont-ils mieux documentés que le reste ?

```python
couverture_notebook_tools = None  # TODO etudiant
```

## Conclusion — ce que ce premier livrable établit

Ce carnet dresse la **carte sourcée** de la surface **invocable par un agent** dans le dépôt : les interfaces **vérifiées** par classe (MCP / CLI / bridge), et — surtout — la **mesure** qui distingue **ce qui existe** de **ce qui est documenté**.

Il répond au **critère 1** de l'EPIC #27071 (*inventaire sourcé de l'existant, distinguant capacités vérifiées / hypothèses / travaux projetés*). Les critères suivants — **pilote CLI** (3), **sémantisation mesurée** (4), **scénarios de bout en bout** (5) — sont les livrables ultérieurs de la série.

**Prochain carnet** : le **pilote** — un script CLI réutilisant un cœur existant, avec succès / échec / sortie structurée, branché sur la surface mesurée ici.

*See #27071.*